In [ ]:
from plot_raster import plot_psth_raster_for_units, plot_raster_and_quantile_psth_by_latent

In [1]:
# ---------------------------------------------------------------------
# EXAMPLE ── integrate with your existing code
# ---------------------------------------------------------------------
import numpy as np
from create_psth import load_zarr
from plot_raster import plot_psth_raster_for_units
from behavior_utils import find_trials


from nwb_utils import NWBUtils
from behavior_utils import extract_fitted_data, find_trials,get_fitted_model_names,generate_behavior_summary
from create_psth import extract_neuron_psth_to_zarr
session_name='764769_2024-12-11_18-21-49'
#session_name='764790_2024-12-19_16-11-34'
print(get_fitted_model_names(session_name=session_name))

nwb_data,tag=NWBUtils.combine_nwb(session_name=session_name)


# 0.  Load the pre‑computed PSTH cube
psth_da = load_zarr("/root/capsule/scratch/psth/ecephys_764769_2024-12-11_18-21-49_sorted_2024-12-13_10-04-48_0.2s.zarr")

# 1.  Pick trials of interest
# 2) convert → numpy array  (enables vectorised −1)
reward=np.asarray(find_trials(nwb_data,'rewarded'))
no_reward=np.asarray(find_trials(nwb_data,'unrewarded'))

# 2.  Plot PSTH for three units over those trials
fig = plot_psth_raster_for_units(
    source=psth_da, 
    align_to_event='go_cue',         
    trial_ids=[[reward],[no_reward]],
    time_window=(-3, 6),
    save_path=None,
    plot_type='mean',
    group_name=['reward','noreward']
)

['QLearning_L2F1_CKfull_softmax', 'QLearning_L2F1_CK1_softmax', 'QLearning_L2F1_softmax', 'WSLS', 'ForagingCompareThreshold', 'QLearning_L1F0_epsi', 'QLearning_L1F1_CK1_softmax']
Found ephys NWB: /root/capsule/data/ecephys_764769_2024-12-11_18-21-49_sorted_2024-12-13_10-04-48/nwb/ecephys_764769_2024-12-11_18-21-49_experiment1_recording1.nwb
Successfully read ephys NWB from: /root/capsule/data/ecephys_764769_2024-12-11_18-21-49_sorted_2024-12-13_10-04-48/nwb/ecephys_764769_2024-12-11_18-21-49_experiment1_recording1.nwb
Found behavior NWB: /root/capsule/data/behavior_nwb/behavior_764769_2024-12-11_18-21-49.nwb
Successfully read behavior NWB from: /root/capsule/data/behavior_nwb/behavior_764769_2024-12-11_18-21-49.nwb
Successfully appended units table to behavior NWB.



KeyboardInterrupt



In [ ]:


from pathlib import Path
import numpy as np

from general_utils import find_ephys_sessions, smart_read_csv
from behavior_utils import find_trials, get_fitted_model_names
from nwb_utils import NWBUtils
from create_psth import load_zarr
from plot_raster import plot_raster_and_quantile_psth_by_latent

PSTH_DIR = Path("/root/capsule/scratch/psth")
RESULTS_DIR = Path("/root/capsule/scratch")
OUTDIR = Path("/root/capsule/scratch/raster_plot")

LATENTS = [
    "QLearning_L2F1_softmax-deltaQ-1",
    "QLearning_L2F1_softmax-reward",
    "QLearning_L2F1_softmax-sumQ-1",
]
latent_names = ['deltaQ-1','reward','sumQ-1']

ALIGN_TO = "go_cue"
TIME_WIN = (-3, 4)


OUTDIR.mkdir(parents=True, exist_ok=True)
_, _, sessions = find_ephys_sessions()
print("Sessions:", sessions)


for s in sessions:
    print(f"\n[{s}] models:", get_fitted_model_names(session_name=s))
    nwb, _ = NWBUtils.combine_nwb(session_name=s)
    psth = load_zarr(str(PSTH_DIR / f"{s}_0.2s.zarr"))
    beh = smart_read_csv(str(RESULTS_DIR / f"behavior_summary-{s}.csv"))
    trials = np.asarray(find_trials(nwb, "response"))
    save_dir = OUTDIR / s
    save_dir.mkdir(exist_ok=True)

    i=0
    for col in LATENTS:
        if col not in beh.columns:
            print(f"  skip: {col} (missing)")
            continue
        vals = beh[col][0]

        plot_raster_and_quantile_psth_by_latent(
            source=psth,
            latent_values=vals,              # full vector
            latent_trial_ids=trials,
            unit_ids=None,
            align_to_event=ALIGN_TO,
            time_window=TIME_WIN,
            n_bins=4,
            binning="equal",
            quantile_stat="mean",
            ci="sem",
            title_prefix=None,
            figsize=(6, 5),
            save_path=str(save_dir),
            cmap_name="coolwarm",
            raster_colormap=False,
            show_colormap=True,
            save_prefix=f'{col}_',
            latent_name=latent_names[i],
            show=False
        )
        i=i+1
        print(f"  plotted: {col}")




In [ ]:
# run_raster_parallel.py
from __future__ import annotations

import os
import multiprocessing as mp
import concurrent.futures as cf
from pathlib import Path

from general_utils import find_ephys_sessions
from raster_worker import process_session, OUTDIR  # import the top-level worker

def main():
    OUTDIR.mkdir(parents=True, exist_ok=True)
    _, _, sessions = find_ephys_sessions()
    print("Sessions:", sessions)
    if not sessions:
        return

    # Use spawn to avoid HDF5/NWB fork-safety issues
    ctx = mp.get_context("spawn")
    max_workers = min(len(sessions), os.cpu_count() or 1)
    print(f"Running in parallel with {max_workers} workers (spawn)")

    with cf.ProcessPoolExecutor(max_workers=max_workers, mp_context=ctx) as ex:
        futures = {ex.submit(process_session, s): s for s in sessions}
        for fut in cf.as_completed(futures):
            print(fut.result())

if __name__ == "__main__":
    # If something else already set the start method, this is fine.
    try:
        mp.set_start_method("spawn", force=False)
    except RuntimeError:
        pass
    main()


In [4]:
# load the correlation results
from general_utils import load_temporary_data

# path that you passed as   save_folder / (save_name + ".csv")
zarr_path = "/root/capsule/scratch/correlation_results/sig_dir_all_sessions.zarr"
ds = load_temporary_data(zarr_path)

print("Rows:", len(ds))
display(ds.head())             # in a notebook


Rows: 69951


,ARDL_model-ForagingCompareThreshold-RPE-g9-s-1-d0-coef,ARDL_model-ForagingCompareThreshold-RPE-g9-s-1-d0-pval,ARDL_model-ForagingCompareThreshold-RPE-g9-s-1-d0-tval,ARDL_model-ForagingCompareThreshold-RPE-g9-s0-d0-coef,ARDL_model-ForagingCompareThreshold-RPE-g9-s0-d0-pval,ARDL_model-ForagingCompareThreshold-RPE-g9-s0-d0-tval,ARDL_model-ForagingCompareThreshold-RPE-g9-s1-d0-coef,ARDL_model-ForagingCompareThreshold-RPE-g9-s1-d0-pval,ARDL_model-ForagingCompareThreshold-RPE-g9-s1-d0-tval,ARDL_model-ForagingCompareThreshold-reward-g2-s-1-d0-coef,...,simple_LR-g9-s1-d0-f_stat,simple_LR-g9-s1-d0-hqic,simple_LR-g9-s1-d0-llf,simple_LR-g9-s1-d0-rsq,simple_LR-g9-s1-d0-rsq_adj,simple_LR-g9-s1-d0-sigma2,source_file,time_window,unit_index,z_score
index,,,,,,,,,,,,,,,,,,,,,
0,0.152402,0.120495,1.556492,0.089148,0.368827,0.899832,-0.020901,0.833563,-0.210290,0.123322,...,2.768431,NaN,-779.059920,0.007695,0.004915,NaN,correlations_multi-ecephys_753124_2024-12-10_1...,-1_0,9,False
1,0.167072,0.079958,1.756028,-0.192590,0.042605,-2.034970,0.118241,0.214108,1.244609,0.123618,...,0.000597,NaN,-681.181162,0.000002,-0.002799,NaN,correlations_multi-ecephys_753124_2024-12-10_1...,0.3_2,9,False
2,0.032352,0.800006,0.253533,-0.185525,0.147099,-1.453067,0.192337,0.132300,1.508614,-0.048639,...,5.735645,NaN,-644.272340,0.015812,0.013055,NaN,correlations_multi-ecephys_753124_2024-12-10_1...,0.3_2_-1_0,9,False
3,0.083825,0.427871,0.793764,-0.033850,0.748948,-0.320279,0.006011,0.954721,0.056821,0.079978,...,1.378883,NaN,-770.762613,0.003848,0.001057,NaN,correlations_multi-ecephys_753124_2024-12-10_1...,-1_0,14,False
4,0.110529,0.181864,1.337697,-0.210355,0.010194,-2.583169,0.095946,0.242588,1.170519,0.210518,...,0.391524,NaN,-681.769576,0.001096,-0.001703,NaN,correlations_multi-ecephys_753124_2024-12-10_1...,0.3_2,14,False


In [3]:
# Define column names
pval_col = "simple_LR-QLearning_L2F1_softmax-reward-g0-s0-d0-pval"
coef_col = "simple_LR-QLearning_L2F1_softmax-reward-g0-s0-d0-coef"

time_window = "0.3_2"
alpha = 0.05
brain_areas = ["SI", "MA"]
#brain_areas = ["MD"]
coef_col_sign = ["negative"]  # can be ["positive"], ["negative"], or both

# Normalize lists
if isinstance(brain_areas, str):
    brain_areas = [brain_areas]

if isinstance(coef_col_sign, str):
    coef_col_sign = [coef_col_sign]

# ---------------------------------------------
# Build the coefficient sign filter
# ---------------------------------------------
sign_mask = True  # default = keep all

if "positive" in coef_col_sign and "negative" not in coef_col_sign:
    sign_mask = ds[coef_col] > 0
elif "negative" in coef_col_sign and "positive" not in coef_col_sign:
    sign_mask = ds[coef_col] < 0
elif "positive" in coef_col_sign and "negative" in coef_col_sign:
    sign_mask = (ds[coef_col] > 0) | (ds[coef_col] < 0)   # i.e., nonzero
else:
    raise ValueError("coef_col_sign must contain 'positive', 'negative', or both.")

# ---------------------------------------------
# Combine masks
# ---------------------------------------------
mask = (
    (ds[pval_col] < alpha) &
    (ds["time_window"] == time_window) &
    (ds["brain_region"].isin(brain_areas)) &
    (sign_mask)
)

# ---------------------------------------------
# Filter rows
# ---------------------------------------------
selected = ds.loc[mask, ["session_id", "unit_index"]]

# Convert formats if needed
result_tuples = list(selected.itertuples(index=False, name=None))
result_dicts = selected.to_dict(orient="records")


NameError: name 'ds' is not defined

In [75]:
result_dicts

[{'session_id': '764769_2024-12-11_18-21-49', 'unit_index': 852},
 {'session_id': '764769_2024-12-11_18-21-49', 'unit_index': 857},
 {'session_id': '764769_2024-12-11_18-21-49', 'unit_index': 877},
 {'session_id': '764769_2024-12-11_18-21-49', 'unit_index': 878},
 {'session_id': '764769_2024-12-11_18-21-49', 'unit_index': 880},
 {'session_id': '764769_2024-12-11_18-21-49', 'unit_index': 886},
 {'session_id': '764769_2024-12-11_18-21-49', 'unit_index': 891},
 {'session_id': '764769_2024-12-11_18-21-49', 'unit_index': 893},
 {'session_id': '764769_2024-12-11_18-21-49', 'unit_index': 899},
 {'session_id': '764769_2024-12-11_18-21-49', 'unit_index': 901},
 {'session_id': '764769_2024-12-11_18-21-49', 'unit_index': 907},
 {'session_id': '764769_2024-12-11_18-21-49', 'unit_index': 912},
 {'session_id': '764769_2024-12-11_18-21-49', 'unit_index': 914},
 {'session_id': '764769_2024-12-11_18-21-49', 'unit_index': 923},
 {'session_id': '764769_2024-12-11_18-21-49', 'unit_index': 930},
 {'session

In [ ]:
# --------------------------------------------------------
# COMPLETE CODE: Loop over multiple result_ids
# --------------------------------------------------------

from pathlib import Path
from PIL import Image
from IPython.display import display

# ------------------------
# Choose the list of result indices
# ------------------------
results_ids = range(len(result_dicts))   # <-- update this list as needed

model_latent = "QLearning_L2F1_CK1_softmax-deltaQ-1"
model_latent = "QLearning_L2F1_CK1_softmax-reward"

base_dir = Path("/root/capsule/scratch/raster_plot")

# ------------------------
# Loop over each result_id
# ------------------------
for rid in results_ids:
    print("\n===============================")
    print(f" Showing result_id = {rid}")
    print("===============================\n")

    session_id = result_dicts[rid]["session_id"]
    unit_id = result_dicts[rid]["unit_index"]

    # ------------------------
    # Locate session folder
    # ------------------------
    pattern = f"ecephys_{session_id}_sorted_*"
    session_folders = list(base_dir.glob(pattern))

    print("Found folders:")
    for f in session_folders:
        print("  ", f)

    if not session_folders:
        print(f"❌ No folder found for session_id: {session_id}")
        continue

    session_folder = session_folders[0]

    # ------------------------
    # Construct figure path
    # ------------------------
    fig_path = session_folder / f"{model_latent}_unit_{unit_id}.png"
    print("Figure path:", fig_path)
    print("Exists?", fig_path.exists())

    if not fig_path.exists():
        print(f"❌ Figure missing for: {fig_path}")
        continue

    # ------------------------
    # Load and resize image
    # ------------------------
    img = Image.open(fig_path)

    new_width = 800
    w, h = img.size
    new_height = int(h * (new_width / w))
    img_resized = img.resize((new_width, new_height), Image.LANCZOS)

    # ------------------------
    # Display resized figure
    # ------------------------
    display(img_resized)
